# Serialized tabular classification on Colab

Select **Runtime → Change runtime type → T4 GPU**. This reproduces the full heldout tabular extension with the frozen Qwen3-4B-Instruct-2507 snapshot: Titanic (262 test rows), Breast Cancer Wisconsin Diagnostic (114), and Wine (36). Each dataset receives zero-shot, four examples per class, and a fixed three-epoch QLoRA run. The shared label budgets are 8, 8, and 12; no development labels select adapters. These small public datasets can be contaminated by pretraining, so results do not establish general tabular superiority.

Prepared rows already contain the task name and named feature serialization. The original classification prompt, numeric-ID-plus-EOS likelihood scoring, seed 42, and data splits are preserved. Training uses rank 8, alpha 16, all-linear targets, zero dropout, response-only loss, NF4, learning rate 0.0002, batch size 1 and accumulation 8. QLoRA evaluation reloads the pinned base at normal CUDA inference precision. Each process exits before the next begins.

This notebook makes no hosted model API calls. It permits public Hugging Face model downloads; Colab compute availability and account charges depend on your account. Upload the private source ZIP and the small transfer helper from the same reviewed repository revision. The ZIP contains public, de-identified feature projections, labels, and attribution; it excludes raw Titanic personal fields.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, stat, tempfile, zipfile

SOURCE_SHA = '0e9d2453273883664c771ad9fe53b7a7cc3a357ffc6ac73ef2803d122c185c4e'
HELPER_SHA = 'c9723b6dd7af5f72363aade43f727df2036cd09506bb694944090676739d706c'
uploaded = files.upload()  # Select jev-tabular-colab.zip AND tabular_colab_artifacts.py
assert set(uploaded) == {'jev-tabular-colab.zip', 'tabular_colab_artifacts.py'}
assert hashlib.sha256(uploaded['jev-tabular-colab.zip']).hexdigest() == SOURCE_SHA
assert hashlib.sha256(uploaded['tabular_colab_artifacts.py']).hexdigest() == HELPER_SHA
work = Path(tempfile.mkdtemp(prefix='jev-tabular-', dir='/content'))
source_zip = work / 'jev-tabular-colab.zip'
source_zip.write_bytes(uploaded['jev-tabular-colab.zip'])
prefix = 'jev-tabular-benchmark/'
with zipfile.ZipFile(source_zip) as archive:
    members = archive.infolist()
    assert len(members) < 1000 and sum(i.file_size for i in members) < 100_000_000
    names = [i.filename for i in members]
    assert len(names) == len(set(names))
    for info in members:
        name = info.filename
        path = PurePosixPath(name)
        assert name.startswith(prefix) and path.as_posix() == name and not path.is_absolute()
        assert '..' not in path.parts and '\\' not in name and ':' not in name
        assert not info.is_dir() and stat.S_IFMT(info.external_attr >> 16) in (0, stat.S_IFREG)
        assert not info.flag_bits & 1 and info.file_size < 30_000_000
    manifest = json.loads(archive.read(prefix + 'bundle_manifest.json'))
    expected = manifest['files_sha256']
    assert set(names) == {prefix + name for name in expected} | {prefix + 'bundle_manifest.json'}
    for name, checksum in expected.items():
        assert hashlib.sha256(archive.read(prefix + name)).hexdigest() == checksum
    for info in members:
        destination = work / info.filename
        assert destination.resolve().is_relative_to(work.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(info))
project = work / 'jev-tabular-benchmark'
(project / 'scripts/tabular_colab_artifacts.py').write_bytes(uploaded['tabular_colab_artifacts.py'])
del uploaded
print('Verified fresh project:', project)


In [ ]:
%cd {project}
%pip install -q -e ".[dev,neural,qlora]"
# Colab can preinstall a torchao version incompatible with current PEFT.
# This benchmark uses bitsandbytes NF4; remove the unused optional torchao package.
%pip uninstall -y torchao


In [ ]:
import importlib.metadata, os, platform, subprocess, sys
import torch
from jevbench.runner import environment
assert torch.cuda.is_available(), 'A CUDA GPU is required; no CPU fallback'
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
assert version('torchao') is None, 'Remove the optional conflicting torchao package before running'
os.environ.setdefault('HF_HOME', '/content/jev-tabular-hf')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
result_folder = project / 'results/tabular/colab'
result_folder.mkdir(parents=True, exist_ok=True)
env = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'cuda_available': True, 'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'cuda_version': torch.version.cuda, 'torchao_version': version('torchao'),
    'packages': {name: version(name) for name in ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'numpy', 'scikit-learn')},
    'core_source_sha256': environment()['source_sha256'],
    'source_bundle_sha256': SOURCE_SHA, 'transfer_helper_sha256': HELPER_SHA,
}
(result_folder / 'environment.json').write_text(json.dumps(env, indent=2, sort_keys=True) + '\n')
print(json.dumps(env, indent=2))
command = [sys.executable, 'scripts/run_tabular_local.py', '--model-key', 'qwen_main',
           '--device', 'cuda', '--load-in-4bit', '--allow-download',
           '--output', 'results/tabular/colab']
subprocess.run(command + ['--phase', 'plan'], cwd=project, check=True)


## Execute the fixed protocol

This runs nine evaluations (1,236 decisions) and three adapter trainings. The helper keeps source/data hashes, execution commands and training provenance. Completed prediction files resume without new decisions; verified completed adapters are reused. A partial nonempty adapter directory or mismatched training recipe causes a stop. Do not alter prompts, seeds, epochs or data after inspecting test outcomes.


In [ ]:
subprocess.run(command + ['--phase', 'all'], cwd=project, check=True)


## Export and verify locally

The transfer includes completed prediction records, environment/plans, dataset/source manifests, and the three adapter configs, LoRA-only weights and training metadata. It excludes raw rows and pretrained base weights. The exporter requires all nine conditions and independently verifies ordered heldout rows, selected training rows, pinned model/core/helper provenance, tensor structure and recomputed metrics.

Keep the printed archive SHA-256. Import into the original local checkout with:

```bash
.venv/bin/python scripts/tabular_colab_artifacts.py import \
  --source-bundle artifacts/jev-tabular-colab.zip \
  --archive /path/to/jev-tabular-results.zip --sha256 PRINTED_SHA256
```

Import preserves the original Colab run records and paths as evidence. Local adapters land under `models/tabular/Qwen3-4B-Instruct-2507/`; an existing differing artifact causes a stop before copying. Repeating an identical verified import is supported.


In [ ]:
export_zip = work / 'jev-tabular-results.zip'
completed = subprocess.run(
    [sys.executable, 'scripts/tabular_colab_artifacts.py', 'export',
     '--source-bundle', str(source_zip), '--output', str(export_zip)],
    cwd=project, check=True, text=True, capture_output=True)
export_report = json.loads(completed.stdout)
print(json.dumps(export_report, indent=2))
files.download(str(export_zip))
